In [37]:
import pandas as pd
import numpy as np

In [38]:
df = pd.read_csv('data/cleaned_online_retail_II.csv')
df['CustomerID'] = df['CustomerID'].astype('Int64')
df.head(5)

,InvoiceNo,InvoiceDate,Date,Year,Month,Day,Quarter,Week,Weekday,DayOfYear,StockCode,CustomerID,Country,Description,UnitPrice,Quantity,Revenue
0,489434,2009-12-01 07:45:00,2009-12-01,2009,12,1,4,49,1,335,21232,13085,United Kingdom,STRAWBERRY CERAMIC TRINKET BOX,1.25,24,30.0
1,489434,2009-12-01 07:45:00,2009-12-01,2009,12,1,4,49,1,335,21523,13085,United Kingdom,FANCY FONT HOME SWEET HOME DOORMAT,5.95,10,59.5
2,489434,2009-12-01 07:45:00,2009-12-01,2009,12,1,4,49,1,335,21871,13085,United Kingdom,SAVE THE PLANET MUG,1.25,24,30.0
3,489434,2009-12-01 07:45:00,2009-12-01,2009,12,1,4,49,1,335,22041,13085,United Kingdom,"RECORD FRAME 7"" SINGLE SIZE",2.10,48,100.8
4,489434,2009-12-01 07:45:00,2009-12-01,2009,12,1,4,49,1,335,22064,13085,United Kingdom,PINK DOUGHNUT TRINKET POT,1.65,24,39.6


In [39]:
def build_daily_features(df_filtered, train_cutoff_date=None, only_business_days=False):
  temporal_cols = ['Date', 'Year', 'Quarter', 'Month', 'Week', 'Weekday', 'Day', 'DayOfYear']
  daily = (
    df_filtered
    .groupby(temporal_cols, as_index=False)
    .agg(Revenue=('Revenue', 'sum'))
  )
  daily['Date'] = pd.to_datetime(daily['Date'])
  daily = daily.sort_values('Date').reset_index(drop=True)

  full_range = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
  daily = daily.set_index('Date').reindex(full_range).rename_axis('Date').reset_index()

  daily['Year']      = daily['Date'].dt.year
  daily['Quarter']   = daily['Date'].dt.quarter
  daily['Month']     = daily['Date'].dt.month
  daily['Week']      = daily['Date'].dt.isocalendar().week.astype(int)
  daily['Weekday']   = daily['Date'].dt.weekday
  daily['Day']       = daily['Date'].dt.day
  daily['DayOfYear'] = daily['Date'].dt.dayofyear
  daily['Revenue']   = daily['Revenue'].fillna(0)

  if only_business_days:
    daily = daily[~daily['Weekday'].isin([5, 6])].reset_index(drop=True)

  if train_cutoff_date is None:
    train_cutoff_date = daily['Date'].max() + pd.Timedelta(days=1)
  train_cutoff_date = pd.Timestamp(train_cutoff_date)
  train_mask = daily['Date'] < train_cutoff_date

  daily['PreChristmas'] = (
    (daily['DayOfYear'] >= 243) & (daily['DayOfYear'] <= 358)
  ).astype(int)

  train_data = daily[train_mask].copy()

  for col in ['Weekday', 'Month', 'Quarter']:
    stats = train_data.groupby(col)['Revenue'].agg(
      mean='mean', median='median',
      q25=lambda x: x.quantile(0.25),
      q75=lambda x: x.quantile(0.75)
    )
    stats['IQR'] = stats['q75'] - stats['q25']
    stats = stats[['mean', 'median', 'IQR']].add_prefix(f'{col}Revenue_')
    daily = daily.merge(stats, on=col, how='left')

  xmas_stats = train_data.groupby('PreChristmas')['Revenue'].agg(
    PreChristmasRevenue_mean='mean',
    PreChristmasRevenue_std='std'
  ).reset_index()
  daily = daily.merge(xmas_stats, on='PreChristmas', how='left')
  daily['PreChristmasRevenue_std'] = daily['PreChristmasRevenue_std'].fillna(0)

  for lag in [7, 14, 30]:
    daily[f'RevenueLag_{lag}d'] = daily['Revenue'].shift(lag)

  lag_cols = [c for c in daily.columns if c.startswith('RevenueLag_')]
  daily.loc[~train_mask, lag_cols] = np.nan

  revenue_shifted = daily['Revenue'].shift(1)
  for window in [7, 30]:
    daily[f'RollingMean_{window}d'] = revenue_shifted.rolling(window, min_periods=1).mean()
    daily[f'RollingStd_{window}d']  = revenue_shifted.rolling(window, min_periods=2).std()
    daily[f'RollingMax_{window}d']  = revenue_shifted.rolling(window, min_periods=1).max()

  for col in [c for c in daily.columns if 'RollingStd' in c]:
    daily[col] = daily[col].fillna(0)

  rolling_cols = [c for c in daily.columns if c.startswith('Rolling')]
  daily.loc[~train_mask, rolling_cols] = np.nan

  cols_to_interpolate = [
    c for c in daily.columns
    if daily[c].isna().any()
    and c not in ['Date']
    and pd.api.types.is_numeric_dtype(daily[c])
  ]
  for col in cols_to_interpolate:
    daily.loc[train_mask, col] = (
      daily.loc[train_mask, col]
      .interpolate(method='linear', limit_direction='both')
    )

  return daily

In [40]:
product = df[df['Description'] == 'REGENCY CAKESTAND 3 TIER']

product_daily = build_daily_features(
  product,
  train_cutoff_date='2011-09-01',
  only_business_days=True
)

price_daily = (
  df[df['Description'] == 'REGENCY CAKESTAND 3 TIER']
  .groupby('Date')['UnitPrice']
  .agg(UnitPrice_median='median', UnitPrice_std='std')
  .reset_index()
)
price_daily['Date'] = pd.to_datetime(price_daily['Date'])

product_daily = product_daily.merge(price_daily, on='Date', how='left')
product_daily['UnitPrice_median'] = (
  product_daily['UnitPrice_median']
  .interpolate(method='linear', limit_direction='both')
)
product_daily['UnitPrice_std'] = product_daily['UnitPrice_std'].fillna(0)

product_daily.to_csv('data/product.csv', index=False)

print(f"Periodo: {product_daily['Date'].min().date()} -> {product_daily['Date'].max().date()}")
print(f"Total de dias: {len(product_daily)}")
print(f"Dias com Revenue > 0: {(product_daily['Revenue'] > 0).sum()}")
print(f"Dias com Revenue = 0: {(product_daily['Revenue'] == 0).sum()}")
print(f"Colunas: {list(product_daily.columns)}")
product_daily.head(10)

Periodo: 2010-03-15 -> 2011-12-08
Total de dias: 454
Dias com Revenue > 0: 420
Dias com Revenue = 0: 34
Colunas: ['Date', 'Year', 'Quarter', 'Month', 'Week', 'Weekday', 'Day', 'DayOfYear', 'Revenue', 'PreChristmas', 'WeekdayRevenue_mean', 'WeekdayRevenue_median', 'WeekdayRevenue_IQR', 'MonthRevenue_mean', 'MonthRevenue_median', 'MonthRevenue_IQR', 'QuarterRevenue_mean', 'QuarterRevenue_median', 'QuarterRevenue_IQR', 'PreChristmasRevenue_mean', 'PreChristmasRevenue_std', 'RevenueLag_7d', 'RevenueLag_14d', 'RevenueLag_30d', 'RollingMean_7d', 'RollingStd_7d', 'RollingMax_7d', 'RollingMean_30d', 'RollingStd_30d', 'RollingMax_30d', 'UnitPrice_median', 'UnitPrice_std']


,Date,Year,Quarter,Month,Week,Weekday,Day,DayOfYear,Revenue,PreChristmas,...,RevenueLag_14d,RevenueLag_30d,RollingMean_7d,RollingStd_7d,RollingMax_7d,RollingMean_30d,RollingStd_30d,RollingMax_30d,UnitPrice_median,UnitPrice_std
0,2010-03-15,2010,1,3,11,0,15,74,102.00,0,...,102.0,102.0,102.000000,0.000000,102.00,102.000000,0.000000,102.00,12.75,0.00
1,2010-03-16,2010,1,3,11,1,16,75,293.25,0,...,102.0,102.0,102.000000,0.000000,102.00,102.000000,0.000000,102.00,12.75,0.00
2,2010-03-17,2010,1,3,11,2,17,76,267.75,0,...,102.0,102.0,197.625000,135.234172,293.25,197.625000,135.234172,293.25,12.75,0.00
3,2010-03-18,2010,1,3,11,3,18,77,267.75,0,...,102.0,102.0,221.000000,103.842730,293.25,221.000000,103.842730,293.25,12.75,0.00
4,2010-03-19,2010,1,3,11,4,19,78,369.75,0,...,102.0,102.0,232.687500,87.950359,293.25,232.687500,87.950359,293.25,12.75,0.00
5,2010-03-22,2010,1,3,12,0,22,81,51.00,0,...,102.0,102.0,260.100000,97.768477,369.75,260.100000,97.768477,369.75,12.75,0.00
6,2010-03-23,2010,1,3,12,1,23,82,216.75,0,...,102.0,102.0,225.250000,122.205053,369.75,225.250000,122.205053,369.75,12.75,0.00
7,2010-03-24,2010,1,3,12,2,24,83,63.75,0,...,102.0,102.0,224.035714,111.603691,369.75,224.035714,111.603691,369.75,12.75,0.00
8,2010-03-25,2010,1,3,12,3,25,84,521.10,0,...,102.0,102.0,218.571429,119.249102,369.75,204.000000,117.845162,369.75,12.75,0.45
9,2010-03-26,2010,1,3,12,4,26,85,229.50,0,...,102.0,102.0,251.121429,165.253451,521.10,239.233333,152.722090,521.10,12.75,0.00


In [41]:
customer = df[df['CustomerID'] == 14911]

customer_daily = build_daily_features(
  customer,
  train_cutoff_date='2011-09-01',
  only_business_days=False
)

sku_daily = (
  df[df['CustomerID'] == 14911]
  .groupby('Date')['StockCode']
  .nunique()
  .reset_index()
  .rename(columns={'StockCode': 'UniqueProducts'})
)
sku_daily['Date'] = pd.to_datetime(sku_daily['Date'])

customer_daily = customer_daily.merge(sku_daily, on='Date', how='left')
customer_daily['UniqueProducts'] = customer_daily['UniqueProducts'].fillna(0)

customer_daily.to_csv('data/customer.csv', index=False)

print(f"Periodo: {customer_daily['Date'].min().date()} -> {customer_daily['Date'].max().date()}")
print(f"Total de dias: {len(customer_daily)}")
print(f"Dias com Revenue > 0: {(customer_daily['Revenue'] > 0).sum()} ({(customer_daily['Revenue'] > 0).mean()*100:.1f}%)")
print(f"Dias com Revenue = 0: {(customer_daily['Revenue'] == 0).sum()} ({(customer_daily['Revenue'] == 0).mean()*100:.1f}%)")
print(f"Colunas: {list(customer_daily.columns)}")
customer_daily.head(10)

Periodo: 2009-12-01 -> 2011-12-08
Total de dias: 738
Dias com Revenue > 0: 246 (33.3%)
Dias com Revenue = 0: 492 (66.7%)
Colunas: ['Date', 'Year', 'Quarter', 'Month', 'Week', 'Weekday', 'Day', 'DayOfYear', 'Revenue', 'PreChristmas', 'WeekdayRevenue_mean', 'WeekdayRevenue_median', 'WeekdayRevenue_IQR', 'MonthRevenue_mean', 'MonthRevenue_median', 'MonthRevenue_IQR', 'QuarterRevenue_mean', 'QuarterRevenue_median', 'QuarterRevenue_IQR', 'PreChristmasRevenue_mean', 'PreChristmasRevenue_std', 'RevenueLag_7d', 'RevenueLag_14d', 'RevenueLag_30d', 'RollingMean_7d', 'RollingStd_7d', 'RollingMax_7d', 'RollingMean_30d', 'RollingStd_30d', 'RollingMax_30d', 'UniqueProducts']


,Date,Year,Quarter,Month,Week,Weekday,Day,DayOfYear,Revenue,PreChristmas,...,RevenueLag_7d,RevenueLag_14d,RevenueLag_30d,RollingMean_7d,RollingStd_7d,RollingMax_7d,RollingMean_30d,RollingStd_30d,RollingMax_30d,UniqueProducts
0,2009-12-01,2009,4,12,49,1,1,335,557.08,1,...,557.08,557.08,557.08,557.080000,0.000000,557.08,557.080000,0.000000,557.08,26.0
1,2009-12-02,2009,4,12,49,2,2,336,0.00,1,...,557.08,557.08,557.08,557.080000,0.000000,557.08,557.080000,0.000000,557.08,0.0
2,2009-12-03,2009,4,12,49,3,3,337,0.00,1,...,557.08,557.08,557.08,278.540000,393.915046,557.08,278.540000,393.915046,557.08,0.0
3,2009-12-04,2009,4,12,49,4,4,338,0.00,1,...,557.08,557.08,557.08,185.693333,321.630288,557.08,185.693333,321.630288,557.08,0.0
4,2009-12-05,2009,4,12,49,5,5,339,0.00,1,...,557.08,557.08,557.08,139.270000,278.540000,557.08,139.270000,278.540000,557.08,0.0
5,2009-12-06,2009,4,12,49,6,6,340,0.00,1,...,557.08,557.08,557.08,111.416000,249.133750,557.08,111.416000,249.133750,557.08,0.0
6,2009-12-07,2009,4,12,50,0,7,341,1257.70,1,...,557.08,557.08,557.08,92.846667,227.426958,557.08,92.846667,227.426958,557.08,45.0
7,2009-12-08,2009,4,12,50,1,8,342,426.15,1,...,557.08,557.08,557.08,259.254286,486.767899,1257.70,259.254286,486.767899,1257.70,12.0
8,2009-12-09,2009,4,12,50,2,9,343,0.00,1,...,0.00,557.08,557.08,240.550000,475.808609,1257.70,280.116250,454.506063,1257.70,0.0
9,2009-12-10,2009,4,12,50,3,10,344,263.34,1,...,0.00,557.08,557.08,240.550000,475.808609,1257.70,248.992222,435.283973,1257.70,13.0


In [42]:
country = df[df['Country'] == 'United Kingdom']

country_daily = build_daily_features(
  country,
  train_cutoff_date='2011-09-01',
  only_business_days=True
)

customers_daily = (
  df[df['Country'] == 'United Kingdom']
  .groupby('Date')['CustomerID']
  .nunique()
  .reset_index()
  .rename(columns={'CustomerID': 'UniqueCustomers'})
)
customers_daily['Date'] = pd.to_datetime(customers_daily['Date'])

country_daily = country_daily.merge(customers_daily, on='Date', how='left')
country_daily['UniqueCustomers'] = country_daily['UniqueCustomers'].fillna(0)

country_daily.to_csv('data/country.csv', index=False)

print(f"Periodo: {country_daily['Date'].min().date()} -> {country_daily['Date'].max().date()}")
print(f"Total de dias: {len(country_daily)}")
print(f"Dias com Revenue > 0: {(country_daily['Revenue'] > 0).sum()}")
print(f"Colunas: {list(country_daily.columns)}")
country_daily.head(10)

Periodo: 2009-12-01 -> 2011-12-09
Total de dias: 529
Dias com Revenue > 0: 504
Colunas: ['Date', 'Year', 'Quarter', 'Month', 'Week', 'Weekday', 'Day', 'DayOfYear', 'Revenue', 'PreChristmas', 'WeekdayRevenue_mean', 'WeekdayRevenue_median', 'WeekdayRevenue_IQR', 'MonthRevenue_mean', 'MonthRevenue_median', 'MonthRevenue_IQR', 'QuarterRevenue_mean', 'QuarterRevenue_median', 'QuarterRevenue_IQR', 'PreChristmasRevenue_mean', 'PreChristmasRevenue_std', 'RevenueLag_7d', 'RevenueLag_14d', 'RevenueLag_30d', 'RollingMean_7d', 'RollingStd_7d', 'RollingMax_7d', 'RollingMean_30d', 'RollingStd_30d', 'RollingMax_30d', 'UniqueCustomers']


,Date,Year,Quarter,Month,Week,Weekday,Day,DayOfYear,Revenue,PreChristmas,...,RevenueLag_7d,RevenueLag_14d,RevenueLag_30d,RollingMean_7d,RollingStd_7d,RollingMax_7d,RollingMean_30d,RollingStd_30d,RollingMax_30d,UniqueCustomers
0,2009-12-01,2009,4,12,49,1,1,335,21298.86,1,...,21298.86,21298.86,21298.86,21298.860000,0.000000,21298.86,21298.860000,0.000000,21298.86,79.0
1,2009-12-02,2009,4,12,49,2,2,336,20808.35,1,...,21298.86,21298.86,21298.86,21298.860000,0.000000,21298.86,21298.860000,0.000000,21298.86,86.0
2,2009-12-03,2009,4,12,49,3,3,337,27590.83,1,...,21298.86,21298.86,21298.86,21053.605000,346.842947,21298.86,21053.605000,346.842947,21298.86,96.0
3,2009-12-04,2009,4,12,49,4,4,338,19471.45,1,...,21298.86,21298.86,21298.86,23232.680000,3782.228653,27590.83,23232.680000,3782.228653,27590.83,69.0
4,2009-12-07,2009,4,12,50,0,7,341,18349.83,1,...,21298.86,21298.86,21298.86,22292.372500,3615.736232,27590.83,22292.372500,3615.736232,27590.83,69.0
5,2009-12-08,2009,4,12,50,1,8,342,22719.30,1,...,21298.86,21298.86,21298.86,21503.864000,3593.590078,27590.83,21503.864000,3593.590078,27590.83,72.0
6,2009-12-09,2009,4,12,50,2,9,343,16452.29,1,...,21298.86,21298.86,21298.86,21706.436667,3252.280097,27590.83,21706.436667,3252.280097,27590.83,67.0
7,2009-12-10,2009,4,12,50,3,10,344,20138.91,1,...,21298.86,21298.86,21298.86,20955.844286,3571.856751,27590.83,20955.844286,3571.856751,27590.83,74.0
8,2009-12-11,2009,4,12,50,4,11,345,15029.21,1,...,20808.35,21298.86,21298.86,20790.137143,3580.187931,27590.83,20853.727500,3319.486251,27590.83,51.0
9,2009-12-14,2009,4,12,51,0,14,348,18359.15,1,...,27590.83,21298.86,21298.86,19964.545714,4189.733785,27590.83,20206.558889,3662.111447,27590.83,62.0
